In [1]:
# 1. Montar Drive & caminhos
from google.colab import drive
import os, subprocess

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else \
       ("MeuDrive" if os.path.exists("/content/drive/MeuDrive") else "MyDrive")
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
PROC_PATH = os.path.join(BASE_PATH, "data_processed")
print("PROC_PATH:", PROC_PATH)

Mounted at /content/drive
PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [2]:
# 2. Carregar dados processados
import pandas as pd, numpy as np

df_ibov = pd.read_csv(os.path.join(PROC_PATH, "ibovespa_clean.csv"))
df_news = pd.read_csv(os.path.join(PROC_PATH, "noticias_clean.csv"))

df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

print("IBOV:", df_ibov.shape, "NEWS:", df_news.shape)
display(df_news.head())

IBOV: (20, 2) NEWS: (10, 6)


,data,fonte,titulo,texto,sentimento,clean_text
0,2024-01-02,exemplo,Titulo 1,Ibovespa sobe com otimismo no mercado internac...,1,ibovespa sobe otimismo mercado internacional
1,2024-01-03,exemplo,Titulo 2,Bolsa cai após dados fracos da economia chinesa.,0,bolsa cai após dados fracos economia chinesa
2,2024-01-04,exemplo,Titulo 3,Petróleo em alta puxa ações da Petrobras para ...,1,petróleo alta puxa ações petrobras cima
3,2024-01-05,exemplo,Titulo 4,Queda do dólar beneficia empresas exportadoras.,1,queda dólar beneficia empresas exportadoras
4,2024-01-08,exemplo,Titulo 5,Setor bancário recua com aversão ao risco.,0,setor bancário recua aversão risco


In [3]:
# 3. Instalar & carregar modelo de embeddings (FinBERT ou BERTimbau)
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

# modelo multilíngue (pt-en) robusto
model = SentenceTransformer("all-MiniLM-L6-v2")

# gerar embeddings para as notícias
embeddings = model.encode(df_news["clean_text"].fillna("").tolist(), show_progress_bar=True)

import pandas as pd
emb_df = pd.DataFrame(embeddings)
emb_df["data"] = df_news["data"].values

# Agregar por dia (média dos embeddings das notícias do dia)
daily_emb = emb_df.groupby("data").mean().reset_index()
print("Daily Embeddings shape:", daily_emb.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Daily Embeddings shape: (10, 385)


In [4]:
# 4. Merge com IBOV e criar target
df = pd.merge(df_ibov.sort_values("data"), daily_emb, on="data", how="left").fillna(0)
df["ret1"] = df["close"].pct_change().shift(-1)           # retorno do dia seguinte
df = df.dropna().copy()
df["y"] = (df["ret1"] > 0).astype(int)

X = df.drop(columns=["data","close","ret1","y"]).values
y = df["y"].values

print("Final dataset:", X.shape, y.shape)

Final dataset: (19, 384) (19,)


In [5]:
# 5. Modelos: Logistic, RandomForest, XGBoost
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8, random_state=42)
}

tscv = TimeSeriesSplit(n_splits=3)
results = {}

for name, clf in models.items():
    aucs, mdas = [], []
    for tr, te in tscv.split(X):
        clf.fit(X[tr], y[tr])
        proba = clf.predict_proba(X[te])[:,1]
        pred  = (proba > 0.5).astype(int)
        aucs.append(roc_auc_score(y[te], proba))
        mdas.append(accuracy_score(y[te], pred))
    results[name] = {"AUC": np.mean(aucs), "MDA": np.mean(mdas)}

print("Resultados médios (Embeddings):")
print(results)

Resultados médios (Embeddings):
{'Logistic': {'AUC': np.float64(0.5555555555555556), 'MDA': np.float64(0.5833333333333334)}, 'RandomForest': {'AUC': np.float64(0.4444444444444445), 'MDA': np.float64(0.6666666666666666)}, 'XGBoost': {'AUC': np.float64(0.5), 'MDA': np.float64(0.5833333333333334)}}
